# Inspect US Market Prices

This notebook inspects the published `cookekieran/us_market_prices` dataset without modifying it. It checks the S&P 500 price coverage, schema, data quality, trading-day gaps, and simple derived returns. `HF_TOKEN` is loaded locally from `.env` and is never stored in the notebook.

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
from huggingface_hub import HfApi, hf_hub_download

REPO_ID = 'cookekieran/us_market_prices'
FILENAME = 'sp500_daily_prices.parquet'

load_dotenv(Path.cwd() / '.env')
HF_TOKEN = os.getenv('HF_TOKEN')
assert HF_TOKEN, 'HF_TOKEN is missing. Add it to the local .env file.'

api = HfApi(token=HF_TOKEN)
revision = api.repo_info(repo_id=REPO_ID, repo_type='dataset').sha
print(f'Repository: {REPO_ID}')
print(f'Pinned revision: {revision}')

## Repository files

In [ ]:
repo_files = api.list_repo_files(repo_id=REPO_ID, repo_type='dataset', revision=revision)
display(pd.DataFrame({'path': repo_files}))

## Load the published S&P 500 prices

In [ ]:
local_path = hf_hub_download(
    repo_id=REPO_ID,
    repo_type='dataset',
    filename=FILENAME,
    revision=revision,
    token=HF_TOKEN,
)

prices = pd.read_parquet(local_path)
prices['date'] = pd.to_datetime(prices['date'])
prices = prices.sort_values(['ticker', 'date']).reset_index(drop=True)

print(f'Rows: {len(prices):,}')
print(f'Columns: {prices.columns.tolist()}')
display(prices.head())

## Coverage and data quality

In [ ]:
coverage = (
    prices.groupby('ticker')
    .agg(first_date=('date', 'min'), last_date=('date', 'max'), rows=('date', 'size'))
    .reset_index()
)
display(coverage)

expected_columns = {'date', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'ticker'}
print('Missing expected columns:', expected_columns - set(prices.columns))
print('Duplicate ticker-date rows:', prices.duplicated(['ticker', 'date']).sum())
display(prices.isna().sum().rename('missing_values').to_frame())

invalid_ohlc = prices[
    (prices['low'] > prices['high'])
    | (prices['open'] < prices['low']) | (prices['open'] > prices['high'])
    | (prices['close'] < prices['low']) | (prices['close'] > prices['high'])
    | (prices['volume'] < 0)
]
print('Invalid OHLC or volume rows:', len(invalid_ohlc))

## Trading-day gaps and derived returns

In [ ]:
prices['calendar_gap_days'] = prices.groupby('ticker')['date'].diff().dt.days
display(prices.loc[prices['calendar_gap_days'] > 4, ['ticker', 'date', 'calendar_gap_days']])

prices['return_1d'] = prices.groupby('ticker')['adj_close'].pct_change()
prices['return_5d'] = prices.groupby('ticker')['adj_close'].pct_change(5)
display(prices[['return_1d', 'return_5d']].describe(percentiles=[0.01, 0.5, 0.99]))

## Price and return overview

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(prices['date'], prices['adj_close'])
axes[0].set(title='S&P 500 adjusted close', ylabel='Adjusted close')

axes[1].plot(prices['date'], prices['return_1d'], linewidth=0.7)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set(title='Daily return', ylabel='Return', xlabel='Date')

plt.tight_layout()